# 

# Notebook to run a comparison of models exported to ONNX files

## Caveats:
* Models have not been retrained in this pt > 0.3 range
* Currently runs on the whole set - not the training / test that might be different 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import importlib
import Utils
import Evaluation

from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix, classification_report, accuracy_score, average_precision_score, log_loss, auc
from hipe4ml.tree_handler import TreeHandler
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import softmax
from scipy.stats import ks_2samp
from sklearn.inspection import permutation_importance
import seaborn as sns
from sklearn.calibration import CalibrationDisplay
from matplotlib import cm


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

import onnx
import numpy as np
from onnx import helper, numpy_helper, TensorProto
import onnxruntime as ort
import Utils
import matplotlib.pyplot as plt
import importlib
import pandas as pd
import seaborn as sns
import Evaluation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
importlib.reload(Utils)
importlib.reload(Evaluation)

In [ ]:
df_OO = Utils.get_dataframe("Data/OOwmatchattempts.root", folder_name="DF_*")
df_PbPb = Utils.get_dataframe("Data/PbPbwmatchattempts.root", folder_name="DF_*")


In [ ]:
df_OO = Utils.process_dataframe(df_OO, makedummies=False)
df_PbPb = Utils.process_dataframe(df_PbPb, makedummies=False)

In [ ]:
# df_OO = Utils.subsample(df_OO, frac = 0.6)
# df_PbPb = Utils.subsample(df_PbPb, frac = 0.6)
# Subsample due to memory conditions, about a third for now

_, _, df_OO = Evaluation.Splitter(df_OO, val_frac=0.1, test_frac = 0.3) 
_, _, df_PbPb = Evaluation.Splitter(df_PbPb, val_frac=0.1, test_frac = 0.3) 
# Grab only the test set for unbiased evaluation


In [ ]:
# FEATURES_OO = ['DeltaDirection', 'PullPt', 'APullPhi', 'PullTanl', 'PtMFT',
#        'CPhiPhiMFT', 'DeltaR', 'SameSign', 'DeltaEta', 'DeltaTanl',
#        'RelPtDiff', 'CYYMFT', 'C1Pt1PtMFT', 'PullPhi', 'CXXMFT',
#        'C1PtPhiMFT', 'YMCH', 'XMCH', 'DeltaPt', 'ADeltaPhi', 'PullR',
#        'PullX', 'PullY', 'CXYMFT', 'CTglTglMCH', 'DeltaPhi', 'C1PtXMFT']

# FEATURES_PBPB = ['RelPtDiff', 'SameSign', 'PtMFT', 'PullPt', 'CPhiPhiMFT',
#        'C1Pt1PtMFT', 'CTglTglMCH', 'CPhiPhiMCH', 'TanlMFT', 'CXXMFT',
#        'CYYMFT', 'DeltaDirection', 'DeltaR', 'ADeltaPhi', 'etaMFT',
#        'PullR', 'DeltaPt', 'C1PtPhiMFT', 'DeltaEta', 'ADeltaX',
#        'APullPhi', 'DeltaTanl', 'CYYMCH', 'ADeltaY', 'CTglTglMFT',
#        'PullTanl', 'CXXMCH', 'InvQPtMFT', 'PtMCH', 'C1Pt1PtMCH', 'PullY',
#        'CXYMFT', 'APullX', 'DeltaX', 'APullY', 'DeltaPhi', 'C1PtXMFT',
#        'DeltaY', 'CTglXMCH', 'etaMCH', 'PullX', 'YMCH', 'TanlMCH', 'XMCH',
#        'CPhiXMFT', 'CTglXMFT', 'CPhiYMFT', 'C1PtYMFT', 'CTglPhiMFT',
#        'PullPhi', 'XMFT', 'YMFT', 'CPhiYMCH', 'CTglYMFT', 'PhiMFT',
#        'PhiMCH', 'C1PtTglMCH', 'CTglYMCH', 'C1PtTglMFT', 'CPhiXMCH']

FEATURES = ['XMCH',
 'YMCH',
 'PhiMCH',
 'TanlMCH',
 'InvQPtMCH',
 'CXXMCH',
 'CYYMCH',
 'CPhiPhiMCH',
 'CTglTglMCH',
 'C1Pt1PtMCH',
 'CXYMCH',
 'CPhiYMCH',
 'CPhiXMCH',
 'CTglXMCH',
 'CTglYMCH',
 'CTglPhiMCH',
 'C1PtXMCH',
 'C1PtYMCH',
 'C1PtPhiMCH',
 'C1PtTglMCH',
 'XMFT',
 'YMFT',
 'PhiMFT',
 'TanlMFT',
 'InvQPtMFT',
 'TrackTypeMFT',
 'CXXMFT',
 'CYYMFT',
 'CPhiPhiMFT',
 'CTglTglMFT',
 'C1Pt1PtMFT',
 'CXYMFT',
 'CPhiYMFT',
 'CPhiXMFT',
 'CTglXMFT',
 'CTglYMFT',
 'CTglPhiMFT',
 'C1PtXMFT',
 'C1PtYMFT',
 'C1PtPhiMFT',
 'C1PtTglMFT',
 'etaMCH',
 'etaMFT',
 'DeltaEta',
 'DeltaX',
 'DeltaY',
 'DeltaPhi',
 'ADeltaPhi',
 'ADeltaX',
 'ADeltaY',
 'DeltaTanl',
 'DeltaR',
 'RMFT',
 'SameSign',
 'PtMCH',
 'PtMFT',
 'DeltaPt',
 'RelPtDiff',
 'PullPt',
 'PullX',
 'PullY',
 'PullR',
 'PullPhi',
 'PullTanl',
 'APullX',
 'APullY',
 'APullPhi',
 'DeltaDirection']

MODEL_OO = "lgbmOOallfeaturespt03.onnx"

MODEL_PBPB = "lgbmpbpballfeaturespt03.onnx"


# Evaluation

In [ ]:
df_OO = Evaluation.onnxinferlgbm(df_OO, FEATURES, MODEL_OO, 'score')

In [ ]:
df_PbPb = Evaluation.onnxinferlgbm(df_PbPb, FEATURES, MODEL_PBPB, 'score')

## PBPB

In [ ]:
Utils.plot_metrics_vs_xy(df_PbPb,
    feature_x="PtMCH",
    # fmin_x = 0.3,
    fmax_x = 3.5,
    feature_y="MatchAttempts",
    # fmin_y = 0.0,
    fmax_y = 5000.0,
    threshold=0.5,
    metrics_fn=Utils.inhousemetrics,
    x_bins=4,
    y_bins=4,
    metric_col_prefix = 'score')
#TODO for presentation: crossection slices for pt & matchattempts respectively - metricwisefeatureplots1D for comparsion of models & systems evaluated on one another
#TODO: add functionality for the ratio of these plots, with teh eventual aim fof comparing different models evaluated on different systems

## OO

In [ ]:
Utils.plot_metrics_vs_xy(df_OO,
    feature_x="PtMCH",
    # fmin_x = 0.3,
    fmax_x = 8,
    feature_y="MatchAttempts",
    # fmin_y = 0.0,
    fmax_y = 300.0,
    threshold=0.6,
    metrics_fn=Utils.inhousemetrics,
    x_bins=4,
    y_bins=4,)

#TODO: score disrtibution decomposition - e.g. momentum, pt, eta, matchattempts --- featurewise - depending on score distributions 
#TODO: Develop a JSOn based output&input for the model evaluation on O2Physics

# Score distributions

In [ ]:
#TODO: revisit the leading match score distributions, and using this, on the validation set, determine a useful manner for optimal seperability

## OO


In [ ]:
df_OO[['MatchAttempts','PtMCH']].describe()


In [ ]:
ooptmed = df_OO['PtMCH'].median()
oomamed = df_OO['MatchAttempts'].median()

### Mult

In [ ]:
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO), feature="score", title="combined score distribution", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO[(df_OO['MatchAttempts']>oomamed) ]), feature="score", title="high mult", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO[(df_OO['MatchAttempts']<oomamed) ]), feature="score", title="low mult", log = True, density = False)
# for entry in Utils.GROUP_PRESERVING_FEATURES:   
#     Utils.plot_metrics_vs_feature(df=df_OO,feature=entry, threshold = 0.8, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=25, trim_low=0.0, trim_high=0., Nsigma=1.0)

# & (df_OO['PtMCH']>ooptmed)

Notice the change in where the distributions cut one another; True&Wrong from 0.8->0.5; wrong+fake & True 0.3-0.5->0.1-0.2

Also note the relatvie number of each group within each plot changes rapidly

### pt

In [ ]:
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO), feature="score", title="combined score distribution", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO[(df_OO['PtMCH']>ooptmed) ]), feature="score", title="high pt", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO[(df_OO['PtMCH']<ooptmed)]), feature="score", title="low pt", log = True, density = False)

distributions continue to remain largely similar, but groups changed prevalence 

## PbPb

In [ ]:
df_PbPb[['MatchAttempts','PtMCH']].describe()

In [ ]:
pbptmed = df_PbPb['PtMCH'].median()
pbmamed = df_PbPb['MatchAttempts'].median() #.quantile(0.25)

### Mult

In [ ]:
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb), feature="score", title="combined score distribution", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb[(df_PbPb['MatchAttempts']>pbmamed) ]), feature="score", title="high mult", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb[(df_PbPb['MatchAttempts']<pbmamed) ]), feature="score", title="low mult", log = True, density = False)


#TODO: add decomposition of pt/MatchAttempts true match score distributions and plot on the same 
# required for determining if we need a matchattempts mult dependent thresholds

Virtually no change based on this multiplicity split

### pt

In [ ]:
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb), feature="score", title="combined score distribution", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb[(df_PbPb['PtMCH']>pbptmed) ]), feature="score", title="high pt", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb[(df_PbPb['PtMCH']<pbptmed)]), feature="score", title="low pt", log = True, density = False)

At low pt we can note a creep of the Wrong matches up to high scores

At low pt we see that the score distribution of True matches is missing its strong peak

# Featurewise metric sweeps

## OO

Overall solid performance, TP~0.97, purity 0.99 at the optimal highptlowmatchattempts bin

In [ ]:
df_oo_highpt = df_OO[df_OO['PtMCH'] > 1.0]
for entry in ['MatchAttempts', 'Rabs', 'PtMCH']:   
    Utils.plot_metrics_vs_feature(df=df_oo_highpt,feature=entry, threshold = 0.8, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=25, trim_low=0.0, trim_high=0.1, Nsigma=1.0)

# NOTE: RABS adds + logmomemtum probably 

## PbPb
At low MatchAttempts in this high pt bin we can retrieve respectable, but not as good as OO, performance with both key metrics > 0.9

NOTE: Still with the missing matches included, so a ~5% degradation is not unexpected

At higher matchattempts we do see the usual degradation, but not as bad as in the combined case

In [ ]:
df_PbPb_highpt = df_PbPb[(df_PbPb['PtMCH'] > 1.0) ]
for entry in ['MatchAttempts', 'Rabs', 'PtMCH']:   
    Utils.plot_metrics_vs_feature(df=df_PbPb,feature=entry, threshold = 0.8, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=50, trim_low=0.0, trim_high=0.1, Nsigma=1.0)

# TODO: for comparisons focus on similar Rabs bins regions as wel
#TODO: retrain smaller models on a specific snippet of featurespace

# The changes we see in the underlying metrics as a function of group preserving variables reflect the fact that we score distributions change as a function of these as well